In [1]:
from pyspark.sql import SparkSession
 
# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("Limpiar Carpetas") \
    .getOrCreate()

In [2]:
import json
from notebookutils import mssparkutils

# CONFIGURACIÓN
RUTA_JSON_ORIGINAL = f"abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/lista_urls.json"
RUTA_RAW = f"abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/"
RUTA_RECHAZADOS = f"abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/rejected/fallos_descarga.json"

print("Auditor en marcha: Buscando archivos fallidos...")

# Leemos la lista original que intentó descargar la pipeline
texto_json = mssparkutils.fs.head(RUTA_JSON_ORIGINAL, 1024000)
lista_esperada = json.loads(texto_json)

archivos_rechazados = []

# DETECTAR LOS FALLOS
for item in lista_esperada:
    anio = item['anio']
    mes = item['mes']
    tipo = item['tipo']
    archivo = item['url_part']
    
    ruta_destino = f"{RUTA_RAW}year={anio}/month={mes}/type={tipo}/{archivo}"
    
    try:
        info = mssparkutils.fs.ls(ruta_destino)[0]
        # Si existe pero está vacío, también lo consideramos un fallo
        if info.size == 0:
            item['motivo_rechazo'] = "Archivo de 0 bytes"
            archivos_rechazados.append(item)
    except:
        # Si da error, significa que el Copy Data falló por el 403
        item['motivo_rechazo'] = "Error de descarga (403/No encontrado)"
        archivos_rechazados.append(item)

# GUARDAR EL REGISTRO EN REJECTED
if archivos_rechazados:
    json_rechazados = json.dumps(archivos_rechazados, indent=2)
    mssparkutils.fs.put(RUTA_RECHAZADOS, json_rechazados, True)
    print(f"Se detectaron {len(archivos_rechazados)} fallos.")
    print(f"Se ha guardado el listado en: {RUTA_RECHAZADOS}")
else:
    print("La pipeline se ejecutó a la perfección. No hay archivos rechazados.")

In [3]:
spark.stop()